## Copula Estimation for the positional marginals

Models the dependence structure of one manager's whole squad with a truncated C-vine
copula over their personalized marginals (from `longitudinal.ipynb`'s
`fit_manager_pipeline` / `alpha_manager`). Split out into its own notebook since it's a
distinct modelling stage with its own dependency (`pyvinecopulib`) and its own data
requirement (enough historical weeks for Kendall's tau to be meaningful -- see the
caveat near the bottom).

Bivariate copula fitting, AIC-based family selection, and vine simulation use
`pyvinecopulib` rather than a from-scratch implementation.

One correction to the write-up: the stated boundary formulas
$a_{t,k}=(1-X_{t,k})(1-p_{t,k})$, $b_{t,k}=X_{t,k}+(1-X_{t,k})(1-p_{t,k})$ collapse to
$a=b=1-p_{t,k}$ when $X_{t,k}=0$ (a zero-width interval, not the stated
$[0, 1-p_{t,k})$). The cases actually given for $u_{t,k}$'s support --
$[0, 1-p_{t,k})$ when not selected, $[1-p_{t,k}, 1]$ when selected -- are unambiguous and
are exactly the standard _distributional transform_ used to continuize a discrete
(here Bernoulli($p_{t,k}$)) margin for copula modelling, so that's what's implemented
below: $a_{t,k} = X_{t,k}\cdot(1-p_{t,k})$, $b_{t,k} = X_{t,k} + (1-X_{t,k})(1-p_{t,k})$.


### Setup

Reloads what this stage needs from `longitudinal.ipynb`: the raw data, the league
fits (for `feats`/`X_mean`/`X_std`/`beta_mean`), and the position-pool/`alpha_manager`
helpers. Doesn't re-run any PyMC fitting -- it loads a `manager_fits_{GW}.joblib`
already produced by `longitudinal.ipynb`'s per-round loop.


In [ ]:
import ast

import numpy as np
import pandas as pd
import joblib
import pyvinecopulib as pv
from scipy.stats import kendalltau

player_data = pd.read_csv('../../rolled_data_24_25.csv')
league_selections = pd.read_csv('../../league_selections_df.csv')

LEAGUE_FITS = {
    'Goalkeeper': joblib.load('../estimates/league_models_gk'),
    'Defender': joblib.load('../estimates/league_models_def'),
    'Midfielder': joblib.load('../estimates/league_models_mid'),
    'Forward': joblib.load('../estimates/league_models_fwd'),
}

POSITION_BINARY_COL = {
    'Goalkeeper': 'goalkeeper_binary',
    'Defender': 'defender_binary',
    'Midfielder': 'midfielder_binary',
    'Forward': 'forward_binary',
}


def get_league_fit_for_round(league_fits, target_round):
    """Pick the league fit at `target_round`, or the most recent one available
    before it (see longitudinal.ipynb for the full rationale)."""
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(f"No league fit available at or before round {target_round}")
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    """The player pool for `position` at gameweek `rnd`, sorted by element id
    ascending -- matches the *_binary columns in league_selections for that round."""
    pool = player_data[(player_data['round'] == rnd) & (player_data['position'] == position)] \
        .sort_values('element')
    elements = pool['element'].to_numpy()
    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw


def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    """alpha_{j,m,t,k} (eq:alpha_m) for a fitted manager model at new raw features
    X_new_raw and retention lag y_prev."""
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = X_std_new @ beta_m + manager_fit["gamma_mean"] * y_prev
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)

In [ ]:
# Parameters -- change these to point at a different manager/gameweek's fit.
# manager_fits_{GW}.joblib is produced by longitudinal.ipynb's per-round loop and
# stores {GW: {position: fit_manager_pipeline_result}}.
GW = 7
TEAM_ID = 38054  # GW7's opponent under longitudinal.ipynb's round-robin fixture (manager 205, seeds 61/16)

manager_fits = joblib.load(f'./estimates/manager_fits_{GW}.joblib')[GW]
print(f"Loaded manager_fits for GW{GW}, team {TEAM_ID}: positions {list(manager_fits.keys())}")

### Restricting to the pool $M$

Per position, rank players by their gameweek-`GW` marginal $p_{j,m,GW,k}$ and keep the
top `N_TOP`, unioned with whichever players the manager actually owns that week (so the
observed squad is always representable in $M$, even if the model didn't rank one of
them highly).


In [ ]:
def get_owned(team_id, position, rnd, elements):
    """Ownership map {element: 0/1} for `position` at round `rnd`, or None if that
    round's selection export doesn't line up with `elements`."""
    rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == rnd)]
    if len(rows) == 0:
        return None
    y = np.asarray(ast.literal_eval(rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float)
    if len(y) != len(elements):
        return None
    return dict(zip(elements.tolist(), y.tolist()))


def position_marginals_at_week(team_id, position, manager_fit, league_fit, rnd):
    """p_{k,rnd} (eq:alpha_m, normalized) for every player in `position`'s pool at
    week `rnd`, plus that week's ownership map (or None if unavailable)."""
    elements, X_raw = _position_pool(player_data, position, rnd, league_fit['feats'])
    owned_prev_map = get_owned(team_id, position, rnd - 1, elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    p = alpha / alpha.sum()
    owned_map = get_owned(team_id, position, rnd, elements)
    return dict(zip(elements.tolist(), p.tolist())), owned_map


N_TOP = 30
pool_records = []
for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    manager_fit = manager_fits[position]
    league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
    p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, GW)

    ranked = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
    top_elements = {e for e, _ in ranked[:N_TOP]}
    owned_elements = {e for e, v in (owned_map or {}).items() if v == 1}
    for e in top_elements | owned_elements:
        pool_records.append({"element": e, "position": position})

pool_df = pd.DataFrame(pool_records).sort_values(['position', 'element']).reset_index(drop=True)
elements_ordered = pool_df['element'].tolist()
M = len(elements_ordered)
print(f"Pool M size: {M}  (by position: {pool_df['position'].value_counts().to_dict()})")

### Data Augmentation

For every historical week $t$ and every pool player $k$, get $X_{t,k}$ (actual
occupancy) and $p_{t,k}$ (that week's `alpha_manager`-derived marginal), then draw the
continuized latent uniform via the distributional transform: $u_{t,k} = a_{t,k} +
U\cdot(b_{t,k}-a_{t,k})$ for $U\sim\text{Uniform}(0,1)$, with $a_{t,k}=X_{t,k}(1-p_{t,k})$,
$b_{t,k}=X_{t,k}+(1-X_{t,k})(1-p_{t,k})$ (see the correction noted above).


In [ ]:
history_rounds = sorted(
    league_selections.loc[
        (league_selections['team_id'] == TEAM_ID) & (league_selections['round'] < GW) & (league_selections['round'] >= 4),
        'round'
    ].unique().tolist()
)
print("history rounds used:", history_rounds)

X_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
P_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)

for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    manager_fit = manager_fits[position]
    league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
    pos_elements = pool_df.loc[pool_df['position'] == position, 'element'].tolist()
    for rnd in history_rounds:
        p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, rnd)
        for e in pos_elements:
            X_hist.loc[rnd, e] = (owned_map or {}).get(e, 0.0)
            # a player not yet in the position pool that week (hasn't debuted/transferred
            # into the league yet) gets a tiny floor probability rather than an undefined one
            P_hist.loc[rnd, e] = p_map.get(e, 1e-6)


def latent_uniforms(X_hist, P_hist, rng):
    """Distributional-transform latent uniforms u_{t,k} (see markdown above)."""
    a = np.where(X_hist.values == 0, 0.0, 1.0 - P_hist.values)
    b = np.where(X_hist.values == 0, 1.0 - P_hist.values, 1.0)
    draws = rng.uniform(size=X_hist.shape)
    U = a + draws * (b - a)
    return pd.DataFrame(U, index=X_hist.index, columns=X_hist.columns)


rng = np.random.default_rng(42)
U = latent_uniforms(X_hist, P_hist, rng)
print("U shape:", U.shape)
U.round(3)

### Permutation vector: Kendall's $\tau$ and Hub Score

$\hat\tau_{i,j}$ (eq. `Eq:kendall_tau`) between every pair of pool players from $U$'s
$T$ rows, then $S_k = \sum_{j\neq k}|\hat\tau_{k,j}|$ (eq. `Eq:hub_score`); sorting by
descending $S_k$ gives the C-vine's root-node order $\pi$.

**Caveat**: this needs enough historical weeks $T$ for Kendall's $\tau$ to mean
anything -- with the default `GW=7` there are only $T=3$ (rounds 4-6), and $\tau$ from
3 points can only take the values $\{-1,-\tfrac13,\tfrac13,1\}$. Expect that case to
show up downstream as most pair-copulas defaulting to independence and the acceptance
rate collapsing to near zero; rerun with a later `GW` (20+) once more history has
accumulated for a substantive fit.


In [ ]:
U_vals = U.values
tau_matrix = np.zeros((M, M))
for i in range(M):
    for j in range(i + 1, M):
        tau, _ = kendalltau(U_vals[:, i], U_vals[:, j])
        tau = 0.0 if np.isnan(tau) else tau
        tau_matrix[i, j] = tau_matrix[j, i] = tau

hub_scores = np.abs(tau_matrix).sum(axis=1)
order_idx = np.argsort(-hub_scores)  # 0-indexed, descending hub score
cvine_order = (order_idx + 1).tolist()  # pyvinecopulib uses 1-indexed variables

print("tau matrix: min", tau_matrix.min(), "max", tau_matrix.max(), "mean |tau|", np.abs(tau_matrix).mean())
print("top-5 hub scores:", hub_scores[order_idx[:5]])

### Truncated C-Vine: pair-copula selection via AIC

Build a `CVineStructure` from the permutation $\pi$, truncated at tree 15 (eq.
`Eq:c_vine_pi`), and fit it with `pyvinecopulib`'s built-in AIC-based family selection
(eq. `log_likelihood` / the AIC comparison) over a standard parametric candidate set.


In [ ]:
TRUNC_LVL = 15

cvine_structure = pv.CVineStructure(order=cvine_order, trunc_lvl=TRUNC_LVL)
cvine_controls = pv.FitControlsVinecop(
    family_set=[pv.BicopFamily.indep, pv.BicopFamily.gaussian, pv.BicopFamily.clayton,
                pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.joe],
    selection_criterion="aic",
    trunc_lvl=TRUNC_LVL,
)

vine = pv.Vinecop.from_data(U_vals, structure=cvine_structure, controls=cvine_controls)

tree1_families = vine.families[0]
n_indep = sum(1 for f in tree1_families if f == pv.BicopFamily.indep)
print(f"Vine dim={vine.dim}, trunc_lvl={vine.trunc_lvl}")
print(f"Tree 1: {n_indep} / {len(tree1_families)} pairs selected 'independence'")
print("AIC:", vine.aic(U_vals))

### Sampling the C-Vine and the Inverse-PIT

Simulate $u_t^*$ from the fitted vine, discretize each player via
$w^*_{t,k} = \mathbb{1}[u_{t,k}^* \geq 1-p_{k,T}]$ using gameweek `GW`'s own marginals,
and keep only samples within the transfer limit $T'$ of the manager's current squad.


In [ ]:
N_SAMPLES = 1000
T_LIMIT = 2

sim_U = vine.simulate(n=N_SAMPLES, seeds=[42])

p_target, current_owned = {}, {}
for position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    manager_fit = manager_fits[position]
    league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
    p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, GW)
    for e in pool_df.loc[pool_df['position'] == position, 'element']:
        p_target[e] = p_map.get(e, 1e-6)
        current_owned[e] = (owned_map or {}).get(e, 0.0)

p_target_vec = np.array([p_target[e] for e in elements_ordered])
current_vec = np.array([current_owned[e] for e in elements_ordered])

w_star = (sim_U >= (1.0 - p_target_vec)[None, :]).astype(int)
transfers = 0.5 * np.abs(w_star - current_vec[None, :]).sum(axis=1)

print(pd.Series(transfers, name="transfers_per_sample").describe())
feasible_mask = transfers <= T_LIMIT
print(f"\nfeasible samples (<= {T_LIMIT} transfers): {feasible_mask.sum()} / {N_SAMPLES}")

feasible_w = w_star[feasible_mask]
feasible_w[:5]

### Reading the result

If `Tree 1` above shows most pairs at `independence` and the feasible-sample count is
near zero, that's not a bug -- it's what happens when $T$ (historical weeks) is too
small relative to $M$ (pool size) for Kendall's $\tau$ to detect real dependence (at
the default `GW=7`, $T=3$, so this is expected). The whole pipeline (pool construction
-> data augmentation -> Kendall's $\tau$/Hub Score -> truncated C-vine + AIC family
selection -> simulation -> inverse-PIT -> feasibility filter) is mechanically correct
and reusable as-is; for a _useful_ copula fit, rerun the parameters cell above with a
later `GW` for the same opponent (e.g. GW20+, once 15+ historical weeks have
accumulated) and the corresponding `manager_fits_{GW}.joblib`.
